# Colab Training Notebook for TinyStoriesZh

This notebook runs the upstream-style repo on Colab with a Google Drive-backed cache and a conservative tokenizer preparation path.

## 1. Mount Google Drive and configure paths

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

REPO_URL = 'https://github.com/picasso250/autoresearch-zh.git'
BRANCH = 'codex/zh-port'
DRIVE_ROOT = Path('/content/drive/MyDrive')
WORKDIR = DRIVE_ROOT / 'colab' / 'autoresearch-zh'
REPO_DIR = WORKDIR / 'repo'
CACHE_DIR = WORKDIR / 'cache'

WORKDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)

print(f'WORKDIR:   {WORKDIR}')
print(f'REPO_DIR:  {REPO_DIR}')
print(f'CACHE_DIR: {CACHE_DIR}')

## 2. Clone or refresh the repo

In [ ]:
import subprocess

def run(cmd, cwd=None):
    print('>', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

print('Repo ready:', REPO_DIR)

## 3. Install runtime dependencies

In [ ]:
import sys
import subprocess
import torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kernels>=0.11.7',
    'pyarrow>=21.0.0',
    'requests>=2.32.0',
    'rustbpe>=0.1.0',
    'tiktoken>=0.11.0',
], check=True)

print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## 4. Prepare repo imports

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR)
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
os.chdir(REPO_DIR)
os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)

print('cwd:', Path.cwd())
print('repo import path ready:', repo_str)

## 5. Download dataset files only

In [ ]:
import prepare

DATASET = 'tinystorieszh'
os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)
prepare.download_data(DATASET)
print('Dataset download step finished.')

## 6. Train tokenizer only

This uses a smaller text budget than the raw `prepare.py` default so Colab is less likely to kill the process.

In [ ]:
import pickle
import time
import torch
import rustbpe
import tiktoken
import prepare

DATASET = 'tinystorieszh'
TOKENIZER_MAX_CHARS = 200_000_000
TOKENIZER_DOC_CAP = 8_000

os.environ['AUTORESEARCH_CACHE_DIR'] = str(CACHE_DIR)

tokenizer_dir = prepare._tokenizer_dir(DATASET)
tokenizer_pkl = os.path.join(tokenizer_dir, 'tokenizer.pkl')
token_bytes_path = os.path.join(tokenizer_dir, 'token_bytes.pt')

if os.path.exists(tokenizer_pkl) and os.path.exists(token_bytes_path):
    print(f'Tokenizer already exists at {tokenizer_dir}')
else:
    os.makedirs(tokenizer_dir, exist_ok=True)
    print(f'Training tokenizer with max_chars={TOKENIZER_MAX_CHARS:,}, doc_cap={TOKENIZER_DOC_CAP:,}')
    t0 = time.time()
    tokenizer = rustbpe.Tokenizer()
    vocab_size_no_special = prepare.VOCAB_SIZE - len(prepare.SPECIAL_TOKENS)
    tokenizer.train_from_iterator(
        prepare.text_iterator(dataset_name=DATASET, max_chars=TOKENIZER_MAX_CHARS, doc_cap=TOKENIZER_DOC_CAP),
        vocab_size_no_special,
        pattern=prepare.SPLIT_PATTERN,
    )

    pattern = tokenizer.get_pattern()
    mergeable_ranks = {bytes(k): v for k, v in tokenizer.get_mergeable_ranks()}
    token_offset = len(mergeable_ranks)
    special_tokens = {name: token_offset + i for i, name in enumerate(prepare.SPECIAL_TOKENS)}
    enc = tiktoken.Encoding(
        name='rustbpe',
        pat_str=pattern,
        mergeable_ranks=mergeable_ranks,
        special_tokens=special_tokens,
    )

    with open(tokenizer_pkl, 'wb') as f:
        pickle.dump(enc, f)

    token_bytes_list = []
    special_set = set(prepare.SPECIAL_TOKENS)
    for token_id in range(enc.n_vocab):
        token_str = enc.decode([token_id])
        token_bytes_list.append(0 if token_str in special_set else len(token_str.encode('utf-8')))
    torch.save(torch.tensor(token_bytes_list, dtype=torch.int32), token_bytes_path)

    with open(os.path.join(tokenizer_dir, 'dataset.txt'), 'w', encoding='utf-8') as f:
        f.write(DATASET + '\n')

    prepare._set_active_dataset(DATASET)
    print(f'Tokenizer ready in {time.time() - t0:.1f}s at {tokenizer_dir}')

print('Tokenizer step finished.')

## 7. Full training run (simple T4 large-model baseline)

In [ ]:
env = os.environ.copy()
subprocess.run([
    sys.executable, 'train_t4.py', '--dataset', 'tinystorieszh',
    '--depth', '10', '--model-dim', '1024',
    '--device-batch-size', '4',
    '--lr-scale', '0.125',
    '--total-batch-size', '65536',
], cwd=REPO_DIR, env=env, check=True)